# CST 4702 Lab 10: Hierarchical Clustering and DBSCAN

## Iris Dataset Notebook

This notebook completes the clustering lab with a full workflow:

- load or create the dataset,
- visualize it first,
- build dendrograms with `scipy.cluster.hierarchy`,
- test all four hierarchical linkage methods,
- tune and run DBSCAN,
- calculate silhouette values for every sample, and
- end with a short written discussion about DBSCAN and silhouette score behavior.

I kept the comments plain and direct so the notebook reads like normal class work instead of a bare script dump.


## Install the libraries used in this notebook

If you are opening this notebook in a clean runtime, run the next cell once.


In [ ]:
%pip install -q pandas scikit-learn scipy matplotlib


## Imports and helper functions

I put the reusable plotting and scoring helpers up front so the analysis cells stay shorter and easier to follow.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.cluster import AgglomerativeClustering, DBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_samples, silhouette_score
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", 30)
pd.set_option("display.precision", 4)
plt.style.use("seaborn-v0_8-whitegrid")


def plot_cluster_map(points_2d, labels, title, axis=None, x_label="Component 1", y_label="Component 2"):
    if axis is None:
        axis = plt.gca()

    unique_labels = sorted(np.unique(labels))
    palette = plt.get_cmap("tab10")
    color_index = 0

    for label in unique_labels:
        mask = labels == label
        if label == -1:
            axis.scatter(
                points_2d[mask, 0],
                points_2d[mask, 1],
                color="black",
                s=45,
                alpha=0.85,
                label="noise",
            )
        else:
            axis.scatter(
                points_2d[mask, 0],
                points_2d[mask, 1],
                color=palette(color_index % 10),
                s=45,
                alpha=0.85,
                label=f"cluster {label}",
            )
            color_index += 1

    axis.set_title(title)
    axis.set_xlabel(x_label)
    axis.set_ylabel(y_label)
    axis.legend()
    axis.grid(alpha=0.25)


def silhouette_details(features, labels):
    unique_labels = np.unique(labels)
    if len(unique_labels) < 2 or len(unique_labels) >= len(labels):
        return np.full(len(labels), np.nan), np.nan

    sample_scores = silhouette_samples(features, labels)
    average_score = silhouette_score(features, labels)
    return sample_scores, float(average_score)


def describe_clustering(features, labels, model_name):
    sample_scores, average_score = silhouette_details(features, labels)
    row = {
        "model": model_name,
        "cluster_count": int(len(set(labels)) - (1 if -1 in labels else 0)),
        "noise_points": int(np.sum(labels == -1)),
        "silhouette_score": average_score,
    }
    return row, sample_scores


def estimate_eps_from_k_distance(sorted_distances):
    x_values = np.arange(len(sorted_distances), dtype=float)
    points = np.column_stack((x_values, sorted_distances))
    start = points[0]
    end = points[-1]
    line_vector = end - start
    line_vector = line_vector / np.linalg.norm(line_vector)

    projected_points = start + np.outer((points - start) @ line_vector, line_vector)
    distances_to_line = np.linalg.norm(points - projected_points, axis=1)
    elbow_index = int(np.argmax(distances_to_line))
    return float(sorted_distances[elbow_index]), elbow_index


## Load the Iris dataset

I picked Iris for the real dataset because it is standard, small enough to inspect carefully, and still has more than two features.  
Per the lab instructions, I scale the original features for clustering and use PCA only to compress the data to two dimensions for plotting.


In [ ]:
from sklearn.datasets import load_iris

iris_bundle = load_iris(as_frame=True)
iris_features = iris_bundle.data.copy()
iris_target = iris_bundle.target.copy()
iris_names = iris_bundle.target_names

scaler = StandardScaler()
X_iris_scaled = scaler.fit_transform(iris_features)

pca = PCA(n_components=2)
X_iris_pca = pca.fit_transform(X_iris_scaled)

print("Original Iris feature shape:", iris_features.shape)
print("Scaled feature shape:", X_iris_scaled.shape)
display(iris_features.head())


## Visualize the dataset with PCA

I only use PCA here for plotting. The clustering models themselves will still train on the original **scaled 4D data**.


In [ ]:
plt.figure(figsize=(8, 6))

for class_id, class_name in enumerate(iris_names):
    mask = iris_target.to_numpy() == class_id
    plt.scatter(
        X_iris_pca[mask, 0],
        X_iris_pca[mask, 1],
        s=55,
        alpha=0.85,
        label=class_name,
    )

plt.title("Iris data projected to 2D with PCA")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.legend()
plt.grid(alpha=0.25)
plt.show()


## Dendrograms on the scaled original features

The dendrograms are built from the scaled 4D feature space, not from the PCA view.  
The large branch split suggests that using **three clusters** is a reasonable choice here.


In [ ]:
linkage_methods = ["single", "complete", "average", "ward"]

fig, axes = plt.subplots(2, 2, figsize=(16, 11))

for axis, method in zip(axes.ravel(), linkage_methods):
    linkage_matrix = linkage(X_iris_scaled, method=method)
    dendrogram(linkage_matrix, ax=axis, no_labels=True, color_threshold=None)
    axis.set_title(f"{method.title()} linkage dendrogram")
    axis.set_xlabel("Sample index")
    axis.set_ylabel("Distance")

plt.tight_layout()
plt.show()


## Hierarchical clustering results

Every model below is trained on the scaled 4D data, and the scatter plots are shown in PCA space just to make the cluster boundaries visible.


In [ ]:
hierarchical_rows_iris = []
hierarchical_sample_scores_iris = {}
hierarchical_labels_iris = {}

fig, axes = plt.subplots(2, 2, figsize=(16, 11))

for axis, method in zip(axes.ravel(), linkage_methods):
    model = AgglomerativeClustering(n_clusters=3, linkage=method)
    labels = model.fit_predict(X_iris_scaled)

    hierarchical_labels_iris[method] = labels
    plot_cluster_map(
        X_iris_pca,
        labels,
        title=f"{method.title()} linkage clusters",
        axis=axis,
        x_label="Principal Component 1",
        y_label="Principal Component 2",
    )

    row, sample_scores = describe_clustering(
        X_iris_scaled,
        labels,
        model_name=f"Hierarchical - {method}",
    )
    hierarchical_rows_iris.append(row)
    hierarchical_sample_scores_iris[f"{method}_sample_silhouette"] = sample_scores

plt.tight_layout()
plt.show()

hierarchical_results_iris = pd.DataFrame(hierarchical_rows_iris).sort_values(
    by="silhouette_score",
    ascending=False,
)
display(hierarchical_results_iris)


## Silhouette values for every sample in the hierarchical runs


In [ ]:
hierarchical_silhouette_df_iris = pd.DataFrame(hierarchical_sample_scores_iris)
display(hierarchical_silhouette_df_iris.head(10))
display(hierarchical_silhouette_df_iris.describe().T[["mean", "min", "max"]])


## DBSCAN parameter search

The Iris dataset has four original dimensions, so the rule of thumb gives `min_samples = 5`.  
Again, I use the K-distance plot for the first `eps` estimate and then test a small range around it.


In [ ]:
min_samples_iris = X_iris_scaled.shape[1] + 1

nn_model_iris = NearestNeighbors(n_neighbors=min_samples_iris)
distances_iris, _ = nn_model_iris.fit(X_iris_scaled).kneighbors(X_iris_scaled)
k_distances_iris = np.sort(distances_iris[:, -1])

suggested_eps_iris, elbow_index_iris = estimate_eps_from_k_distance(k_distances_iris)

plt.figure(figsize=(9, 5))
plt.plot(k_distances_iris, color="darkorange")
plt.scatter(
    elbow_index_iris,
    k_distances_iris[elbow_index_iris],
    color="crimson",
    s=70,
    label=f"Suggested eps = {suggested_eps_iris:.3f}",
)
plt.title("Iris K-distance graph")
plt.xlabel("Points sorted by distance")
plt.ylabel(f"{min_samples_iris}-NN distance")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

eps_candidates_iris = np.unique(
    np.round(
        np.linspace(
            max(0.10, suggested_eps_iris * 0.55),
            suggested_eps_iris * 1.45,
            10,
        ),
        3,
    )
)

search_rows_iris = []
for eps in eps_candidates_iris:
    labels = DBSCAN(eps=float(eps), min_samples=min_samples_iris).fit_predict(X_iris_scaled)
    row, _ = describe_clustering(X_iris_scaled, labels, model_name=f"DBSCAN eps={eps}")
    row["eps"] = float(eps)
    search_rows_iris.append(row)

dbscan_search_iris = pd.DataFrame(search_rows_iris)
display(dbscan_search_iris.sort_values(by="eps"))


In [ ]:
valid_dbscan_iris = dbscan_search_iris.dropna(subset=["silhouette_score"]).copy()
valid_dbscan_iris = valid_dbscan_iris[valid_dbscan_iris["cluster_count"] >= 2].copy()

best_dbscan_iris = valid_dbscan_iris.sort_values(
    by=["silhouette_score", "noise_points"],
    ascending=[False, True],
).iloc[0]

best_eps_iris = float(best_dbscan_iris["eps"])
dbscan_iris = DBSCAN(eps=best_eps_iris, min_samples=min_samples_iris)
dbscan_iris_labels = dbscan_iris.fit_predict(X_iris_scaled)

dbscan_iris_row, dbscan_iris_sample_scores = describe_clustering(
    X_iris_scaled,
    dbscan_iris_labels,
    model_name=f"DBSCAN (eps={best_eps_iris:.3f}, min_samples={min_samples_iris})",
)

plt.figure(figsize=(8, 6))
plot_cluster_map(
    X_iris_pca,
    dbscan_iris_labels,
    title=f"DBSCAN clusters on Iris (eps={best_eps_iris:.3f})",
    x_label="Principal Component 1",
    y_label="Principal Component 2",
)
plt.show()

print("Chosen eps:", round(best_eps_iris, 3))
print("Total noise points found:", dbscan_iris_row["noise_points"])

dbscan_iris_silhouette_df = pd.DataFrame(
    {
        "cluster_label": dbscan_iris_labels,
        "sample_silhouette": dbscan_iris_sample_scores,
    }
)

display(pd.DataFrame([dbscan_iris_row]))
display(dbscan_iris_silhouette_df.head(10))
display(dbscan_iris_silhouette_df["sample_silhouette"].describe())


## Final comparison


In [ ]:
iris_comparison = pd.concat(
    [
        hierarchical_results_iris,
        pd.DataFrame([dbscan_iris_row]),
    ],
    ignore_index=True,
).sort_values(by="silhouette_score", ascending=False)

display(iris_comparison)


## Short discussion: why DBSCAN can look right but still score lower

A visually good DBSCAN result does not always win on silhouette score because silhouette assumes clusters should be internally tight and cleanly separated from the next group.  
DBSCAN is designed for density-based structure, so it can keep curved shapes, border points, and noise that make visual sense but do not look as compact to the metric.  
In other words, the eye can reward natural shape, while silhouette tends to reward compact geometry.
